# **Day 57: Decorators (Part 2) - wraps and Decorators with Arguments**

## **functools.wraps**
`@wraps(func)` copies the original function's **name** and **docstring** onto the wrapper. We should always use it when making decorators.

In [1]:
from functools import wraps
import time

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"[{func.__name__}] {time.perf_counter() - start:.5f} seconds")
        return result
    return wrapper

@timer
def area(r):
    """Area of a circle"""
    return 3.14159 * r * r

print(area.__name__, "|", area.__doc__)

area | Area of a circle


## **Decorators That Take Their Own Arguments**
Sometimes we want to give settings to a decorator, like `@repeat(3)`. For this we need **one more layer** of function:

```
@repeat(3)
def dhak(): ...

is the same as:  dhak = repeat(3)(dhak)
```
1. `repeat(3)` returns the real decorator.
2. The real decorator receives `dhak` and returns the wrapper.

In [2]:
def repeat(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def dhak():
    print("Dhak kur kur", end=" | ")

dhak()

Dhak kur kur | Dhak kur kur | Dhak kur kur | 

## **Example from Remote Sensing: A Retry Decorator for Downloads**
Downloading satellite images from the internet often fails because of a slow connection or a busy server. Instead of giving up, we should **try again** a few times. A **retry decorator** does this for any function.

In [3]:
import random

def retry(times=3, exceptions=(Exception,), delay=0.0):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    print(f"  Attempt {attempt} failed: {e}")
                    if attempt == times:
                        raise             # Give up after the last attempt
                    time.sleep(delay)
        return wrapper
    return decorator

random.seed(7)

@retry(times=5, exceptions=(ConnectionError,))
def download_scene(scene_id):
    if random.random() < 0.6:                     # Pretend the server is busy 60% of the time
        raise ConnectionError("Server busy")
    return f"{scene_id} downloaded (800 MB)"

print(download_scene("S2B_45QXF_20240120"))

  Attempt 1 failed: Server busy
  Attempt 2 failed: Server busy
S2B_45QXF_20240120 downloaded (800 MB)


Let us understand the code above:
- `retry(times=5, ...)` makes a decorator that will try up to 5 times.
- If `download_scene` raises a `ConnectionError`, the wrapper catches it, prints a message, and tries again.
- The download failed twice and succeeded on the third try.

## **A Validation Decorator**

In [4]:
def must_be_positive(*names):
    def decorator(func):
        @wraps(func)
        def wrapper(**kwargs):
            for name in names:
                if kwargs.get(name, 0) <= 0:
                    raise ValueError(f"{name} must be positive, got {kwargs.get(name)}")
            return func(**kwargs)
        return wrapper
    return decorator

@must_be_positive("principal", "years")
def emi(principal, rate, years):
    r = rate / 12 / 100
    n = years * 12
    return round(principal * r * (1 + r) ** n / ((1 + r) ** n - 1), 2)

print("Monthly EMI:", emi(principal=500000, rate=8.5, years=5))
try:
    emi(principal=500000, rate=8.5, years=0)
except ValueError as e:
    print("Error:", e)

Monthly EMI: 10258.27
Error: years must be positive, got 0


## **Decorating a Class: A Registry**
A decorator can also be used on a class. Here we automatically keep a list of all sensor classes.

In [5]:
SENSORS = {}

def register(cls):
    SENSORS[cls.__name__.lower()] = cls
    return cls

@register
class Landsat8:
    resolution = 30

@register
class Sentinel2:
    resolution = 10

print(SENSORS)
print("Sentinel-2 resolution:", SENSORS["sentinel2"].resolution)

{'landsat8': <class '__main__.Landsat8'>, 'sentinel2': <class '__main__.Sentinel2'>}
Sentinel-2 resolution: 10


This "registry" idea is used by web frameworks. For example, Flask's `@app.route("/")` (Day 95) works the same way.

## **Practice Problem: A Cache That Expires**

In [6]:
def cache_for(seconds):
    def decorator(func):
        store = {}
        @wraps(func)
        def wrapper(*args):
            now = time.time()
            if args in store and now - store[args][1] < seconds:
                print("  (from cache)")
                return store[args][0]
            value = func(*args)
            store[args] = (value, now)
            return value
        return wrapper
    return decorator

@cache_for(1)
def gold_rate(city):
    print("  Fetching a fresh rate...")
    return {"Kolkata": 7250}.get(city, 7200)

print(gold_rate("Kolkata"))
print(gold_rate("Kolkata"))
time.sleep(1.1)
print(gold_rate("Kolkata"))

  Fetching a fresh rate...
7250
  (from cache)
7250


  Fetching a fresh rate...
7250


In [7]:
# Quick question
# How many layers of functions does a decorator with arguments need?
# def deco(x):
#     def real(f):
#         def wrapper(): ...
#         return wrapper
#     return real